# Gold Layer - KPI Analysis

This notebook computes business KPIs from the curated dataset stored in the Gold layer.

Load Data

In [0]:
from pyspark.sql.functions import *

curated = spark.table("e_commerce.gold.gold_curated_orders")

## KPIs

In [0]:
curated.createOrReplaceTempView("curated_view")

1.Total Revenue by Month

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_revenue_by_month AS
SELECT 
    YEAR(order_purchase_timestamp) AS year,
    MONTH(order_purchase_timestamp) AS month,
    SUM(total_payment) AS total_revenue
FROM curated_view
GROUP BY YEAR(order_purchase_timestamp), MONTH(order_purchase_timestamp)
ORDER BY year, month;


SELECT * FROM e_commerce.gold.kpi_revenue_by_month;

2.AOV

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_aov AS
SELECT AVG(order_value) AS AOV
FROM (
    SELECT order_id, MAX(total_payment) AS order_value
    FROM curated_view
    GROUP BY order_id
);

SELECT * FROM e_commerce.gold.kpi_aov;

3. Top 10 products by revenue

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_top_products AS
SELECT 
    product_id,
    product_category_name,
    SUM(price) AS revenue
FROM curated_view
GROUP BY product_id, product_category_name
ORDER BY revenue DESC
LIMIT 10;

SELECT * FROM e_commerce.gold.kpi_top_products;

4. Top 10 sellers by revenue


In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_top_sellers AS
SELECT 
    seller_id,
    SUM(price) AS revenue
FROM curated_view
GROUP BY seller_id
ORDER BY revenue DESC
LIMIT 10;

SELECT * FROM e_commerce.gold.kpi_top_sellers;

5. Orders by customer city and state

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_orders_by_city AS
SELECT 
    customer_city,
    customer_state,
    COUNT(DISTINCT order_id) AS total_orders
FROM curated_view
GROUP BY customer_city, customer_state;

SELECT * FROM e_commerce.gold.kpi_orders_by_city;


6. New vs returning customers (monthly)


In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_new_vs_returning AS
WITH first_orders AS (
    SELECT 
        customer_id,
        MIN(order_purchase_timestamp) AS first_order_date
    FROM curated_view
    GROUP BY customer_id
),
orders_distinct AS (
    SELECT DISTINCT 
        order_id,
        customer_id,
        order_purchase_timestamp
    FROM curated_view
)

SELECT 
    YEAR(o.order_purchase_timestamp) AS year,
    MONTH(o.order_purchase_timestamp) AS month,
    CASE 
        WHEN o.order_purchase_timestamp = f.first_order_date THEN 'New'
        ELSE 'Returning'
    END AS customer_type,
    COUNT(*) AS total_orders
FROM orders_distinct o
JOIN first_orders f
ON o.customer_id = f.customer_id
GROUP BY 
    YEAR(o.order_purchase_timestamp),
    MONTH(o.order_purchase_timestamp),
    customer_type;

SELECT * FROM e_commerce.gold.kpi_new_vs_returning;

7. Customer Lifetime Value (CLV)

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_clv AS
SELECT 
    customer_id,
    SUM(order_value) AS clv
FROM (
    SELECT order_id, customer_id, MAX(total_payment) AS order_value
    FROM curated_view
    GROUP BY order_id, customer_id
)
GROUP BY customer_id;

SELECT * FROM e_commerce.gold.kpi_clv;

8. Order fulfillment rate

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_fulfillment_rate AS
SELECT 
    AVG(is_delivered) AS fulfillment_rate
FROM (
    SELECT DISTINCT order_id, is_delivered
    FROM curated_view
);

SELECT * FROM e_commerce.gold.kpi_fulfillment_rate;

9. Late delivery percentage

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_late_delivery AS
SELECT 
    AVG(is_late_delivery) AS late_delivery_percentage
FROM (
    SELECT DISTINCT order_id, is_late_delivery
    FROM curated_view
);

SELECT * FROM e_commerce.gold.kpi_late_delivery;

10. Average delivery time (days)

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_delivery_time AS
SELECT 
    AVG(delivery_time_days) AS avg_delivery_time_days
FROM (
    SELECT DISTINCT order_id, delivery_time_days
    FROM curated_view
);

SELECT * FROM e_commerce.gold.kpi_delivery_time;

11. Payment method distribution

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_payment_distribution AS
SELECT 
    payment_type,
    COUNT(*) AS total_transactions
FROM e_commerce.gold.fact_payments
GROUP BY payment_type;

SELECT * FROM e_commerce.gold.kpi_payment_distribution;

12. Average review score by product category


In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_review_by_category AS
SELECT 
    product_category_name,
    AVG(review_score) AS avg_review_score
FROM curated_view
GROUP BY product_category_name;

SELECT * FROM e_commerce.gold.kpi_review_by_category;

13. Average review score by seller

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_review_by_seller AS
SELECT 
    seller_id,
    AVG(review_score) AS avg_review_score
FROM curated_view
GROUP BY seller_id;

SELECT * FROM e_commerce.gold.kpi_review_by_seller;

14. Percentage of low-rated orders (rating < 3)

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_low_rating AS
SELECT 
    AVG(COALESCE(is_low_rating, 0)) AS low_rating_percentage
FROM (
    SELECT DISTINCT order_id, is_low_rating
    FROM curated_view
);

SELECT * FROM e_commerce.gold.kpi_low_rating;

## Advanced Analytics - Data Cube

DATA CUBE 1 — Revenue (Time Analysis)

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_revenue_cube AS
SELECT 
    YEAR(order_purchase_timestamp) AS year,
    MONTH(order_purchase_timestamp) AS month,
    SUM(total_payment) AS total_revenue
FROM (
    SELECT order_id, MAX(total_payment) AS total_payment, 
           MAX(order_purchase_timestamp) AS order_purchase_timestamp
    FROM curated_view
    GROUP BY order_id
)
GROUP BY CUBE(
    YEAR(order_purchase_timestamp),
    MONTH(order_purchase_timestamp)
)
ORDER BY year, month;

SELECT * FROM e_commerce.gold.kpi_revenue_cube;

DATA CUBE 2 — Revenue by State & Category

In [0]:
%sql
CREATE OR REPLACE TABLE e_commerce.gold.kpi_state_category_cube AS
SELECT 
    customer_state,
    product_category_name,
    SUM(price) AS total_revenue
FROM curated_view
GROUP BY CUBE(customer_state, product_category_name);

SELECT * FROM e_commerce.gold.kpi_state_category_cube;